# AEI last version
$(\omega - m\Omega_\theta)^2 = \Omega_r^2 + \frac{2B_0 |k|}{\Sigma r} + \frac{k^2 c_s^2}{r^2}$

- $B_0$ campo imperturbato, prpendicolare al disco. legato al materiale del disco - assumiamo legge di potenza con esponente 5/4 come da Shakura&Sunyaev
- $\Sigma$ densità superficiale, legge di potenza con esponente ..
- assumo disco sottile, con aspect ratio $H/r = HOR$
- $c_s$ velocità del suono - in disco sottile è dato approssimativamente dalla formula sotto
- k = numero d'onda. lo prendiamo come parametro liberto ma poi controlliamo che stia in range ragionevole ($k \approx 1/H$ con H scala verticale tipica del disco)
- m: generalmente il modo eccitato ha un solo braccio, perciò poniamo m=1 per ridurre il numro di parametri


> $B_0 = B_{00} (r/r_{H})^{-\alpha_B}$. $B_{00}$ valore di riferimento sull'orizzonte degli eventi ($r_H$) del BH

> $\Sigma = \Sigma_0 (r/r_{in})^{-\alpha_\Sigma}$. $\Sigma_0$ riferimento al borndo interno del disco

> $c_s \approx (H/r) v_\varphi = (H/r) r \Omega_\varphi$

> $0.1/r < |k| < 10/r \ ,\ $ 
> $\beta = \frac{ 8 \pi p}{B^2_0} \approx \frac{ 8 \pi \Sigma c_s^2}{B^2_0} \leq 1 \ ,\ $
> $\frac{d}{dr}\left( \frac{\Omega\Sigma}{B^2_0} \right) > 0$

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from aei_2.aei_setup import (
    solve_k_aei, compute_beta, compute_dQdr,
    r_ilr, r_olr, r_corotation, _make_interp,
    make_disk, get_resonances,
    ALP_B, ALP_S, CONFIGS , A_GRID, B00_GRID, 
    SIGMA0_GRID, B00_REF, SIGMA0_REF, N_R, N_R_CAV
)
from aei_2.cavity_plots import (
    plot_paramspace_heatmap, plot_dQdr_at_CR,
    plot_wkb_heatmap_comparison
)

from setup import M_BH, NU0, r_isco, Rg_SUN, set_style, fix_spines
from aei_2.simple_disc import disk_model_simple

In [ ]:
from matplotlib.colors import LogNorm
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import LogLocator, LogFormatterSciNotation
import matplotlib.cm as cm
from matplotlib.lines import Line2D
from matplotlib.cm import ScalarMappable
import matplotlib.gridspec as gridspec

set_style()

## simple

In [ ]:
def dQdr_profile(r_vec, a, B00, Sigma0, hor):
    """
    Compute dQ/dr = d/dr [Omega_phi * Sigma / B0²].
    The sign is independent of B00, Sigma0 (positive multiplicative constant).
    H/r (hor) does not enter Q at all.
    """
    res = make_disk(r_vec, a, B00, Sigma0, hor)
    B0i = _make_interp(r_vec, res[0])
    Si  = _make_interp(r_vec, res[1])
    return compute_dQdr(r_vec, a, B0i, Si, M_BH)

# ═══════════════════════════════════════════════════════════════════════════════
# 2 ── MAIN SCAN
# ═══════════════════════════════════════════════════════════════════════════════

def cavity_scan(mm, hor, verbose=False):
    """
    Three-step filter over the full (a, B00, Sigma0) grid.

    Step A  dQ/dr > 0 for ALL r ∈ [CR-eps, CR+eps]
            Computed ONCE per (a, mm) — independent of B00, Sigma0, hor.
            The B00/Sigma0 loop is entirely skipped for failing spins.

    Step B  beta(r) ≤ 1 for ALL r ∈ [ISCO, OLR]
            Depends on Sigma0/B00² and hor.

    Step C  1 ≤ k(r) ≤ 1/hr(r) for ALL r ∈ [ISCO, ILR]
            Solves the AEI dispersion relation point by point.

    Returns
    -------
    pd.DataFrame with one row per (a, B00, Sigma0).
    Columns: a, B00, Sigma0, r_ILR, r_OLR, r_CR,
             dQdr_CR, pass_shear, pass_beta, pass_wkb,
             beta_max, k_med, k_min_val, k_max_val, frac_wkb_ok
    """
    records = []
    n_spin = len(A_GRID)
    n_par  = len(B00_GRID) * len(SIGMA0_GRID)

    for i_a, a_val in enumerate(A_GRID):
        rISCO = float(r_isco(a_val))
        r_ILR, r_OLR, r_CR = get_resonances(a_val, mm)

        if verbose:
            print(f'  spin {i_a+1}/{n_spin}  a={a_val:.2f} '
                  f'| r_ILR={r_ILR:.1f}  r_OLR={r_OLR:.1f}  r_CR={r_CR:.1f}',
                  end='  ')

        # ── sanity check on resonance radii ───────────────────────────────
        if not (np.isfinite(r_OLR) and np.isfinite(r_ILR)):
            if verbose:
                print('SKIP (resonances not found)')
            # fill records with failed rows
            for B00 in B00_GRID:
                for S0 in SIGMA0_GRID:
                    records.append(dict(a=a_val, B00=B00, Sigma0=S0,
                                        r_ILR=np.nan, r_OLR=np.nan, r_CR=r_CR,
                                        dQdr_CR=np.nan,
                                        pass_shear=False, pass_beta=False,
                                        pass_wkb=False))
            continue

        if r_ILR <= rISCO * 1.001:
            if verbose:
                print('SKIP (ILR inside ISCO)')
            for B00 in B00_GRID:
                for S0 in SIGMA0_GRID:
                    records.append(dict(a=a_val, B00=B00, Sigma0=S0,
                                        r_ILR=r_ILR, r_OLR=r_OLR, r_CR=r_CR,
                                        dQdr_CR=np.nan,
                                        pass_shear=False, pass_beta=False,
                                        pass_wkb=False))
            continue

        # ── Step A: dQ/dr (independent of B00, Sigma0, hor) ───────────────
        r_olr_cap = min(r_OLR, rISCO * 500.0)
        r_full    = np.geomspace(rISCO*1.001, r_olr_cap, N_R)

        r_cr_eps = np.linspace(r_CR*0.95, r_CR*1.05, 20)
        dq_full  = dQdr_profile(r_cr_eps, a_val, B00_REF, SIGMA0_REF, hor)
        dqdr_at_CR = float(np.interp(r_CR, r_cr_eps, dq_full))
        pass_shear = bool(dqdr_at_CR > 0)

        if verbose:
            shear_str = 'SHEAR OK' if pass_shear else 'shear FAIL'
            print(shear_str)

        if not pass_shear:
            for B00 in B00_GRID:
                for S0 in SIGMA0_GRID:
                    records.append(dict(a=a_val, B00=B00, Sigma0=S0,
                                        r_ILR=r_ILR, r_OLR=r_OLR, r_CR=r_CR,
                                        dQdr_CR=dqdr_at_CR,
                                        pass_shear=False, pass_beta=False,
                                        pass_wkb=False))
            continue

        # ── radial grids for beta & cavity ─────────────────────────────────
        r_ilr_cap = min(r_ILR*0.98, r_olr_cap)
        r_cavity  = np.geomspace(rISCO*1.01, r_ilr_cap, N_R_CAV)

        # ── Steps B & C: loop over (B00, Sigma0) ──────────────────────────
        for B00 in B00_GRID:
            for S0 in SIGMA0_GRID:

                # ── Step B: beta ≤ 1 in [ISCO, OLR] ─────────────────────
                res_full = make_disk(r_full, a_val, B00, S0, hor)
                hr_full  = res_full[3]   # per-point H/r from model
                beta_full = compute_beta(res_full[0], res_full[1], res_full[2],
                                         r_full, hr_full, M_BH)
                pass_beta = bool(np.all(beta_full <= 1.0))
                beta_max  = float(np.max(beta_full))

                if not pass_beta:
                    records.append(dict(a=a_val, B00=B00, Sigma0=S0,
                                        r_ILR=r_ILR, r_OLR=r_OLR, r_CR=r_CR,
                                        dQdr_CR=dqdr_at_CR,
                                        pass_shear=True, pass_beta=False,
                                        pass_wkb=False, beta_max=beta_max))
                    continue

                # ── Step C: WKB in cavity [ISCO, ILR] ────────────────────
                res_cav  = make_disk(r_cavity, a_val, B00, S0, hor)
                B0_c, Sg_c, cs_c, hr_c = res_cav[:4]

                k_arr = solve_k_aei(r_cavity, a_val, B0_c, Sg_c, cs_c,
                                    m=mm, M=M_BH)
                k_max_arr = 1 / np.maximum(hr_c, 1e-10)  # 1/hr(r)

                wkb_ok   = (np.isfinite(k_arr)
                            & (k_arr >= mm)
                            & (k_arr <= k_max_arr))
                pass_wkb = bool(wkb_ok.mean() >= 0.95)
                frac_ok  = float(wkb_ok.sum() / len(wkb_ok))

                k_valid = k_arr[np.isfinite(k_arr)]
                records.append(dict(
                    a=a_val, B00=B00, Sigma0=S0,
                    r_ILR=r_ILR, r_OLR=r_OLR, r_CR=r_CR,
                    dQdr_CR=dqdr_at_CR,
                    pass_shear=True, pass_beta=True, pass_wkb=pass_wkb,
                    beta_max=beta_max,
                    frac_wkb_ok=frac_ok,
                    k_med=float(np.nanmedian(k_arr)),
                    k_min_val=float(np.nanmin(k_arr)) if k_valid.size else np.nan,
                    k_max_val=float(np.nanmax(k_arr)) if k_valid.size else np.nan,
                ))

    return pd.DataFrame(records)


# ═══════════════════════════════════════════════════════════════════════════════
# 3 ── RUN ALL CONFIGS
# ═══════════════════════════════════════════════════════════════════════════════

def run_all():
    """Run cavity_scan for all 4 (m, H/r) configurations."""
    all_results = {}
    for cfg in CONFIGS:
        mm, hor = cfg['mm'], cfg['hor']
        print(f"\n{'='*65}")
        print(f"  Config: m={mm}, H/r={hor}")
        print(f"{'='*65}")
        df = cavity_scan(mm, hor, verbose=False)
        all_results[(mm, hor)] = df

        n      = len(df)
        n_sh   = df['pass_shear'].sum()
        n_be   = df.get('pass_beta',  pd.Series(dtype=bool)).sum() if 'pass_beta' in df else 0
        n_wkb  = df.get('pass_wkb',   pd.Series(dtype=bool)).sum() if 'pass_wkb'  in df else 0
        print(f'\n  Total combos : {n}')
        print(f'  Pass shear   : {n_sh:>6}  ({100*n_sh/n:.1f}%)')
        print(f'  + beta ≤ 1   : {n_be:>6}  ({100*n_be/n:.1f}%)')
        print(f'  + WKB cavity : {n_wkb:>6}  ({100*n_wkb/n:.1f}%)')

    return all_results


def identify_worst(all_results):
    """Return the CONFIGS entry with the lowest fraction of WKB-valid solutions."""
    fracs = {}
    for cfg in CONFIGS:
        key = (cfg['mm'], cfg['hor'])
        df  = all_results[key]
        fracs[key] = df['pass_wkb'].sum() / len(df) if len(df) > 0 else 1.0
    worst_key = min(fracs, key=fracs.get)
    worst_cfg = next(c for c in CONFIGS if (c['mm'], c['hor']) == worst_key)
    print(f'\n  Worst config: m={worst_cfg["mm"]}, H/r={worst_cfg["hor"]}'
          f'  (valid fraction = {fracs[worst_key]:.3f})')
    return worst_cfg

In [ ]:
# ── step 1: full scan ─────────────────────────────────────────────────────
print('\n> Running cavity scan for all configs...')
all_results = run_all()

In [ ]:
cfg = CONFIGS[1]
mm, hor = cfg['mm'], cfg['hor']
df = all_results[(mm, hor)]

stages = [
    ('pass_shear', r'$dQ/dr > 0$ at $r_{\rm CR}$'),
    ('pass_beta',  r'$+\ \beta \leq 1$ everywhere'),
    ('pass_wkb',   r'$+$ WKB in cavity'),
]

fig = plt.figure(figsize=(7, 2.2))
gs = gridspec.GridSpec(1, 4, figure=fig,
                    width_ratios=[1, 1, 1, 0.08],
                    wspace=0.05)
axes = [fig.add_subplot(gs[0, i]) for i in range(3)]
cax  = fig.add_subplot(gs[0, 3])   # asse dedicato alla colorbar

for ax, (col, title) in zip(axes, stages):
    fix_spines(ax)
    # ── build heatmap matrix ───────────────────────────────────────────
    frac_map = np.full((len(B00_GRID), len(SIGMA0_GRID)), np.nan)
    for i, B00 in enumerate(B00_GRID):
        for j, S0 in enumerate(SIGMA0_GRID):
            sub = df[(df['B00'] == B00) & (df['Sigma0'] == S0)]
            if len(sub) == 0:
                continue
            if col not in df.columns:
                frac_map[i, j] = 0.0
                continue
            frac_map[i, j] = sub[col].sum() / len(sub)

    # ── plot ───────────────────────────────────────────────────────────
    im = ax.pcolormesh(SIGMA0_GRID, B00_GRID, frac_map,
                        vmin=0, vmax=1, cmap='RdYlGn', shading='auto')

    # ── reference lines and cross ──────────────────────────────────────
    ax.axvline(SIGMA0_REF, color='white', ls='--', lw=1, alpha=0.85)
    ax.axhline(B00_REF,   color='white', ls='--', lw=1, alpha=0.85)
    ax.plot(SIGMA0_REF, B00_REF, 'x',
            color='white', ms=8, mew=1.5, zorder=3)

    # annotation: total fraction for this filter
    if col in df.columns:
        n_pass = df[col].sum()
        n_tot  = len(df)
        ax.text(0.97, 0.03,
                f'{n_pass}/{n_tot} ({100*n_pass/n_tot:.1f}%)',
                transform=ax.transAxes, va='bottom',
                color='black', horizontalalignment='right',
                bbox=dict(boxstyle='round', fc='white', alpha=0.5))

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\Sigma_0$  [g/cm²]')
    ax.set_title(title)

    # Nascondi le etichette y dei pannelli interni
    if col != 'pass_shear':
        ax.tick_params(labelleft=False)
        ax.sharey(axes[0])

axes[0].set_ylabel(r'$B_{00}$  [G]')

# Colorbar sull'asse dedicato → non tocca la larghezza dei pannelli
norm = Normalize(vmin=0, vmax=1)
cbar = fig.colorbar(ScalarMappable(norm=norm, cmap='RdYlGn'), cax=cax)
cbar.set_label("Frequency [Hz]")

plt.tight_layout()
for ext in ('pdf', 'png'):
    plt.savefig(f'aei_2/simple_heatmap_{hor}_{mm}.{ext}',
                bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
mm=2
cfgs_m = [c for c in CONFIGS if c['mm'] == mm]

fig = plt.figure(figsize=(7,2.8))
gs = gridspec.GridSpec(1, 3, figure=fig,
                    width_ratios=[1, 1, 0.08],
                    wspace=0.05)
axes = [fig.add_subplot(gs[0, i]) for i in range(2)]
cax  = fig.add_subplot(gs[0, 2])   # asse dedicato alla colorbar

# shared colour scale 0→1
vmin, vmax = 0.0, 1.0

for ax, cfg in zip(axes, cfgs_m):
    fix_spines(ax)
    hor = cfg['hor']
    key = (mm, hor)

    if key not in all_results:
        ax.text(0.5, 0.5, f'Dati mancanti\n(mm={mm}, hor={hor})',
                ha='center', va='center', transform=ax.transAxes)
        continue

    df = all_results[key]

    # ── build heatmap ──────────────────────────────────────────────────
    frac_map = np.full((len(B00_GRID), len(SIGMA0_GRID)), np.nan)
    for i, B00 in enumerate(B00_GRID):
        for j, S0 in enumerate(SIGMA0_GRID):
            sub = df[(df['B00'] == B00) & (df['Sigma0'] == S0)]
            if len(sub) == 0:
                continue
            frac_map[i, j] = sub['pass_wkb'].sum() / len(sub)

    n_valid_cells = int(np.sum(frac_map > 0))
    n_cells       = frac_map.size

    im = ax.pcolormesh(
        SIGMA0_GRID, B00_GRID, frac_map,
        vmin=vmin, vmax=vmax,
        cmap='RdYlGn', shading='auto',
    )

    # ── reference lines and cross ──────────────────────────────────────
    ax.axvline(SIGMA0_REF, color='white', ls='--', lw=1, alpha=0.9)
    ax.axhline(B00_REF,   color='white', ls='--', lw=1, alpha=0.9)
    ax.plot(SIGMA0_REF, B00_REF, 'x',
            color='white', ms=8, mew=1.5, zorder=4)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$\Sigma_0$  [g/cm²]')

    ax.text(0.97, 0.03,
        rf'$H/r = {hor}$',
        transform=ax.transAxes, va='bottom',
        horizontalalignment='right',
        color='black',
        bbox=dict(boxstyle='round', fc='white', alpha=0.5))

    # Nascondi le etichette y dei pannelli interni
    if hor == 0.001:
        ax.tick_params(labelleft=False)
        ax.sharey(axes[0])

axes[0].set_ylabel(r'$B_{00}$  [G]')

# Colorbar sull'asse dedicato → non tocca la larghezza dei pannelli
norm = Normalize(vmin=0, vmax=1)
cbar = fig.colorbar(ScalarMappable(norm=norm, cmap='RdYlGn'), cax=cax)
cbar.set_label("Frequency [Hz]")

plt.tight_layout()
for ext in ('pdf', 'png'):
    plt.savefig(f'aei_2/wkb_heatmap_comparison.{ext}', bbox_inches='tight', dpi=150)
plt.show()